# 11 · SinDDM baseline: reduced-budget training + sampling (Phase 8)

**T4 GPU runtime.** Trains SinDDM on the 3 OOD images (128 px) and samples it at several training
lengths, giving a **quality-vs-training-time curve** (the full 120k-step schedule would take ~24 T4-hours
per image, notebook 05).

- Reduced schedule: `STEPS = 20000` with the LR milestones compressed in proportion (3, 7, 12, 13, 15, 18 ×1000).
- **Resumable:** checkpoints are on Drive every 10 min, and `model-1..4.pt` at 5k/10k/15k/20k steps.
  Each session trains for at most `SESSION_MIN` minutes; rerun this notebook until all 3 images show 20000 steps.
- Sampling: 30 images per checkpoint (seeds 10..39) stored in the standard output layout, plus the 3 **official pretrained**
  models (forest, night_sky, pyramids; full training) as the converged reference.
- Evaluate afterwards with notebook 13 (CPU runtime).

In [ ]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK     = "11_sinddm_train"
IMAGES       = ["sd_b0", "sd_fp_house2", "sd_pat_p4"]   # data/sinddm/*.png, 128 px
STEPS        = 20000
SAVE_EVERY   = 5000
SESSION_MIN  = 100                  # training minutes this session (all images together)
SAMPLE_AFTER = [1, 2, 4]            # milestones to sample: 5k, 10k, 20k steps (1..4 = 5k..20k)
N_SAMPLES    = 30

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU (then rerun)."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| bf16 native:", torch.cuda.get_device_capability(0) >= (8, 0))

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
SINDDM = "/content/SinDDM"
if not os.path.exists(SINDDM):
    subprocess.run(["git", "clone", "-q", "https://github.com/fallenshock/SinDDM", SINDDM], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", SINDDM_COMMIT], check=True)
shutil.copytree(f"{PROJECT}/patches/sinddm_port", SINDDM, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("CHANGES.diff"))
!pip install -q einops ftfy scikit-image
print(subprocess.run(["git", "-C", SINDDM, "status", "--short"], capture_output=True, text=True).stdout)

## Inputs

In [ ]:
import torch, collections, time, glob
DATA = "/content/sinddm_data"
for i in IMAGES:
    os.makedirs(f"{DATA}/{i}", exist_ok=True)
    shutil.copy(f"{PROJECT}/data/sinddm/{i}.png", f"{DATA}/{i}/{i}.png")
CK = f"{DRIVE_ROOT}/checkpoints/sinddm"

def trained_steps(i):
    p = f"{CK}/{i}/{i}/model-last.pt"
    if not os.path.exists(p):
        return 0
    with torch.serialization.safe_globals([collections.Counter]):
        return int(torch.load(p, map_location="cpu", weights_only=True)["step"])

print({i: trained_steps(i) for i in IMAGES})

## Train (resumes where the last session stopped; stops cleanly at the session budget)

In [ ]:
remaining = float(SESSION_MIN)
for i in IMAGES:
    done = trained_steps(i)
    if done >= STEPS:
        print(i, "already at", done); continue
    if remaining < 3:
        print("session budget used up; rerun later"); break
    cmd = (f"python main.py --scope {i} --mode train --dataset_folder {DATA}/{i}/ --image_name {i}.png "
           f"--results_folder {CK} --train_num_steps {STEPS + 1} --save_and_sample_every {SAVE_EVERY} "
           f"--sched_k_milestones 3 7 12 13 15 18 --resume --time_budget_min {remaining:.1f} "
           f"--checkpoint_every_min 10 --seed 0 --skip_final_sample 2>&1 | grep -E 'port|step:' | tail -4")
    t0 = time.time(); print(f"--- {i}: from step {done}")
    os.chdir(SINDDM)
    !{cmd}
    remaining -= (time.time() - t0) / 60
    print(i, "now at step", trained_steps(i), f"| {remaining:.0f} session-min left")

## Sample the checkpoints that exist (30 images each) into the standard output layout

In [ ]:
OUT = f"{DRIVE_ROOT}/outputs"
def collect(job, scope, data_dir, image_name, results_folder, milestone, image_id, image_path):
    cmd = (f"python {PROJECT}/scripts/sinddm_collect.py --sinddm-root {SINDDM} --scope {scope} --dataset-folder {data_dir} "
           f"--image-name {image_name} --results-folder {results_folder} --milestone {milestone} --out-root {OUT} "
           f"--job {job} --image-id {image_id} --image-path {image_path} --n {N_SAMPLES} --batch 10 2>&1 | tail -2")
    !{cmd}

for i in IMAGES:
    for k in SAMPLE_AFTER:
        if os.path.exists(f"{CK}/{i}/{i}/model-{k}.pt"):
            collect(f"sinddm_steps{k * SAVE_EVERY}", i, f"{DATA}/{i}/", f"{i}.png", CK, k, i, f"data/sinddm/{i}.png")
        else:
            print(i, "milestone", k, "not trained yet")

## The 3 official pretrained models (full training) as the converged reference

In [ ]:
for scope in ("forest", "night_sky", "pyramids"):
    collect("sinddm_pretrained_full", scope, f"{SINDDM}/datasets/{scope}/", f"{scope}.png", f"{SINDDM}/results", "pretrained",
            scope, f"{SINDDM}/datasets/{scope}/{scope}.png")

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")